In [0]:
from pyspark.sql.functions import *

In [0]:

from pyspark.sql.types import *
df1=spark.read.table("ridestream_cata.bronze.bulk_rides")

display(df1.schema)

In [0]:
df=spark.read.table("ridestream_cata.bronze.read_topic")
df_parsed=df.withColumn("parsed_rides",from_json(col("rides"),rides_schema))\
    .select("parsed_rides.*")

display(df_parsed)

In [0]:
jinja_config = [
    {
        "table" : "ridestream_cata.bronze.stg_rides stg_rides",
        "select" : "stg_rides.*",
        "where" : ""
    },
    {
        "table" : "ridestream_cata.bronze.map_vehicle_makes map_vehicle_makes",
        "select" : "map_vehicle_makes.vehicle_make",
        "where" : "",
        "on" : "stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id"
    },
    {
        "table" : "ridestream_cata.bronze.map_vehicle_types map_vehicle_types",
        "select" : "map_vehicle_types.vehicle_type,map_vehicle_types.description,map_vehicle_types.base_rate,map_vehicle_types.per_mile,map_vehicle_types.per_minute",
        "where" : "",
        "on" : "stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id"
    },
    {
        "table" : "ridestream_cata.bronze.map_ride_statuses map_ride_statuses",
        "select" : "map_ride_statuses.ride_status",
        "where" : "",
        "on" : "stg_rides.ride_status_id = map_ride_statuses.ride_status_id"
    },
    {
        "table" : "ridestream_cata.bronze.map_payment_methods map_payment_methods",
        "select" : "map_payment_methods.payment_method, map_payment_methods.is_card, map_payment_methods.requires_auth",
        "where" : "",
        "on" : "stg_rides.payment_method_id = map_payment_methods.payment_method_id"
    },
    {
        "table" : "ridestream_cata.bronze.map_cities map_cities",
        "select" : "map_cities.city as pickup_city, map_cities.state, map_cities.region",
        "where" : "",
        "on" : "stg_rides.pickup_city_id = map_cities.city_id"
    },
    {
        "table" : "ridestream_cata.bronze.map_cancellation_reasons map_cancellation_reasons",
        "select" : "map_cancellation_reasons.cancellation_reason",
        "where" : "",
        "on" : "stg_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id"
    }
]

In [0]:
from jinja2 import Template

jinja_str="""

SELECT 
{% for i in jinja_config %}
    {{i.select}}
    {% if not loop.last %}
    ,
    {% endif %}
{% endfor %}
FROM
{% for i in  jinja_config %}
    {% if loop.first %}
        {{i.table}}
    {% else %}
        LEFT JOIN {{i.table}}
        ON {{i.on}}
    {% endif %}
{% endfor %}

{% for i in jinja_config %}
    {% if i.where != "" %}
        WHERE {{i.where}}
    {% endif %}
{% endfor %}
    
"""

In [0]:
template=Template(jinja_str)
template_render=template.render(jinja_config=jinja_config)
print(template_render)

In [0]:
%sql


SELECT 
    stg_rides.*,
    map_vehicle_makes.vehicle_make,
    map_vehicle_types.vehicle_type,map_vehicle_types.description,map_vehicle_types.base_rate,map_vehicle_types.per_mile,map_vehicle_types.per_minute,
    map_ride_statuses.ride_status,
    map_payment_methods.payment_method, map_payment_methods.is_card, map_payment_methods.requires_auth,
    map_cities.city as pickup_city, map_cities.state, map_cities.region,
    map_cancellation_reasons.cancellation_reason
FROM
    ridestream_cata.bronze.stg_rides stg_rides
    LEFT JOIN ridestream_cata.bronze.map_vehicle_makes map_vehicle_makes ON stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id
    LEFT JOIN ridestream_cata.bronze.map_vehicle_types map_vehicle_types ON stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id
    LEFT JOIN ridestream_cata.bronze.map_ride_statuses map_ride_statuses ON stg_rides.ride_status_id = map_ride_statuses.ride_status_id
    LEFT JOIN ridestream_cata.bronze.map_payment_methods map_payment_methods ON stg_rides.payment_method_id = map_payment_methods.payment_method_id
    LEFT JOIN ridestream_cata.bronze.map_cities map_cities ON stg_rides.pickup_city_id = map_cities.city_id
    LEFT JOIN ridestream_cata.bronze.map_cancellation_reasons map_cancellation_reasons ON stg_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id
    



    

    

    

    

    

    

    

    

In [0]:
%sql
select * from ridestream_cata.bronze.read_topic